In [1]:
from utils import *
from max_cover import *
from max_cut import *
from max_cut_weighted import *
from imm import *
from knapsack_imm import knapsack_greedy
from heuristic_description import heuristic_description

/home/grads/a/anath/anaconda3/envs/eoh/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


_StoreAction(option_strings=['--iterations'], dest='iterations', nargs=None, const=None, default=10, type=<class 'int'>, choices=None, required=False, help='Number of feature search iterations', metavar=None, deprecated=False)

In [4]:


problem = "Maximum Coverage"
budget = 100
dataset = "HK"
iterations = 1

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
train_graph = load_from_pickle(f'../snap_dataset/train/{dataset}')
val_graph = load_from_pickle(f'../snap_dataset/val/{dataset}')

In [7]:
explainer_feedback_list = []
history = []
best_score = float('-inf')

save_folder = f"{problem}/{dataset}"
os.makedirs(save_folder, exist_ok=True)

model_save_path = os.path.join(save_folder, "best_model.pth")
best_model_data_path = os.path.join(save_folder, "best_model_data.pkl")
history_path = os.path.join(save_folder, "history.pkl")

In [8]:
for iter in tqdm(range(iterations)):
    print(f"Feature Space Search Iteration {iter+1} for problem: {problem}")

    cumulative_feedback = "\n".join(explainer_feedback_list) if explainer_feedback_list else None

    if cumulative_feedback:

        summary_prompt = generate_summary_prompt(cumulative_feedback= cumulative_feedback)
        summary = get_response(client, summary_prompt)
    else:
        summary = None


    node_feature_prompt = generate_llm_prompt(
        problem = problem,
        heuristic_description=heuristic_description[problem],
        problem_definition = problem_definitions[problem],
        explainer_feedback = summary
    )
    
    proposed_features = get_response(client,node_feature_prompt)

    try:
        features, definitions, reasons = parse_llm_features(proposed_features)
        print(f"Proposed features: {features}")
    except Exception as e:
        print(f"Error parsing LLM features: {e}")
        continue



  0%|          | 0/1 [00:00<?, ?it/s]

Feature Space Search Iteration 1 for problem: Maximum Coverage


100%|██████████| 1/1 [00:46<00:00, 46.57s/it]

Proposed features: ['degree', 'closed_neighborhood_size', 'degree_percentile_global', 'degree_percentile_component', 'component_size', 'is_isolated', 'leaf_neighbor_count', 'hub_neighbor_ratio', 'neighbor_degree_mean', 'neighbor_degree_std', 'neighbor_degree_max', 'clustering_coefficient', 'core_number', 'page_rank', 'eigenvector_centrality', 'betweenness_centrality', 'neighborhood_edge_density', 'average_jaccard_with_neighbors', 'overlap_sum_with_neighbors', 'redundancy_index', 'min_jaccard_with_top_k_by_degree', 'mean_jaccard_with_top_k_by_degree', 'unique_coverage_vs_top_k_by_degree', 'distance_to_nearest_hub', 'two_hop_reach', 'neighbor_neighborhood_union_size', 'k_scaled_closed_neighborhood', 'within_component_rank', 'bridge_endpoint_flag', 'local_set_cover_proxy']


In [9]:
definitions

{'degree': 'Number of neighbors of v (|N(v)|).',
 'closed_neighborhood_size': 'Size of v’s closed neighborhood (|N[v]| = 1 + |N(v)|).',
 'degree_percentile_global': 'Percentile rank of v’s degree among all nodes in V.',
 'degree_percentile_component': 'Percentile rank of v’s degree within its connected component.',
 'component_size': 'Number of nodes in v’s connected component.',
 'is_isolated': 'Indicator 1 if degree(v) = 0, else 0.',
 'leaf_neighbor_count': 'Number of neighbors u of v with degree(u) = 1.',
 'hub_neighbor_ratio': 'Fraction of neighbors u with degree(u) ≥ 95th-percentile degree in V.',
 'neighbor_degree_mean': 'Average degree of v’s neighbors.',
 'neighbor_degree_std': 'Standard deviation of degrees of v’s neighbors.',
 'neighbor_degree_max': 'Maximum degree among v’s neighbors.',
 'clustering_coefficient': 'Fraction of possible edges among neighbors of v that actually exist.',
 'core_number': 'k-core index of v (maximum k such that v is in the k-core).',
 'page_rank':

In [ ]:

timeout = 5 # seconds
class TimeoutException(Exception):
        pass

def handler(signum, frame):
    raise TimeoutException

signal.signal(signal.SIGALRM, handler)

train_X = []
codes = {}


if problem.endswith('Weighted'):
    graph_description = (
        f"The input is a weighted NetworkX graph `G` where each node has an attribute `'weight'`, "
        f"and an integer variable `budget` is provided.\n"
    )
    additional_description = "If the feature involves weight, use the existing `'weight'` attribute directly without recomputing it from other functions"
else:
    graph_description = (
        f"The input is a NetworkX graph `G` and the graph is undirected."
       
    )
    additional_description = ''

# Add tqdm to loop
for idx, feature in enumerate(tqdm(features, desc="Extracting features", unit="feature")):
    prompt_code = (
        f"{graph_description}"
        f"Feature name: '{feature}'\n"
        # f"Feature definition: '{definitions[feature]}'\n"
        # f"Write Python code for a function `extract_feature(G, budget)` that computes this feature for all nodes in `G`. "
        f"Write Python code for a function `extract_feature(G)` that computes this feature for all nodes in `G`. "
        f"{additional_description}"
        f"If the budget is relevant to the computation, incorporate it. "
        f"The function should return a NumPy array with the computed feature values.\n"
        f"Ensure the code is efficient and avoids expensive computations.\n"
        f"DO NOT INCLUDE ANY EXPLANATIONS OR COMMENTS.\n"
    )



    start = time.time()
    code_response = get_response(client,prompt_code)
    code = clean_code_block(code_response)

    print(f"Code for feature '{feature}':\n{code}\n")

    
    end = time.time()
    # print(f"Code for feature '{feature}' generated in {end - start:.2f} seconds")

    try:
        # print(f"Extracting feature '{feature}'")
        signal.alarm(timeout)  # Set timeout

        namespace = {}
        exec(code, namespace)  # Execute code in namespace

        # namespace["extract_feature"](G=test_graph, budget=budget)
        feature_values = namespace["extract_feature"](G=train_graph)



        if  isinstance(feature_values, np.ndarray) and feature_values.shape[0] == train_graph.number_of_nodes():
            # train_X[feature] = feature_values
            train_X.append(feature_values)

            codes[feature] = code
            # codes.append(code)
    except (TimeoutException, Exception) as e:

        
        print(f"⚠️ Skipping feature '{feature}' due to error: {e}")

        print('*'*30)
        print(code)
        print('*'*30)
    finally:
        signal.alarm(0)  # Reset alarm


    break



Extracting features:   0%|          | 0/30 [00:00<?, ?feature/s]

Code for feature 'degree':
import numpy as np

def extract_feature(G):
    n = G.number_of_nodes()
    if n == 0:
        return np.empty(0, dtype=np.int64)
    return np.fromiter((G.degree(u) for u in G.nodes()), dtype=np.int64, count=n)



Extracting features:   0%|          | 0/30 [00:07<?, ?feature/s]

⚠️ Skipping feature 'degree' due to error: list indices must be integers or slices, not str
******************************
import numpy as np

def extract_feature(G):
    n = G.number_of_nodes()
    if n == 0:
        return np.empty(0, dtype=np.int64)
    return np.fromiter((G.degree(u) for u in G.nodes()), dtype=np.int64, count=n)
******************************


In [16]:
namespace = {}
exec(code, namespace)  # Execute code in namespace

# namespace["extract_feature"](G=test_graph, budget=budget)
feature_values = namespace["extract_feature"](G=train_graph)

In [18]:
if  isinstance(feature_values, np.ndarray) and feature_values.shape[0] == train_graph.number_of_nodes():
    # train_X[feature] = feature_values
    train_X.append(feature_values)

    codes[feature] = code

TypeError: list indices must be integers or slices, not str